<a href="https://colab.research.google.com/github/JuanZapa7a/Medical-Image-Processing/blob/main/PIM_Challenge/PIM_Challenge_Student_Practice_10.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# UPCT Medical Image Segmentation Challenge 2026-27
## Practice 10

**Course:** Medical Image Processing (521104007)

**Professor:** Juan Zapata

> **New** content of this practice. Copy the cells below and paste them **at the end** of your own notebook (the one you started in Practice 6) — do not repeat the previous practices, you already did them there.
>
> **Final submission:** once you paste these cells, your notebook will contain the complete project (Practices 6-10). Rename it as `PIM_Challenge_Student_apellido1_apellido2_nombre.ipynb` (replace with your real surnames and name) and upload it to the virtual classroom together with your `submission.csv` (or `submission_TTA.csv`, the best one you have uploaded) — the same file you already uploaded to the Kaggle Leaderboard.

## Session Guide (2 hours per session)
| Practice | Dates (Group A / B) | Session Objective | Visual Checkpoint |
|----------|----------------------|-----------------------|-------------------|
| **P6** | 28 Oct - 2 Nov | EDA, Dataset and RLE format | 6 images with masks + RLE OK |
| **P7** | 9-11 Nov | U-Net Baseline and 1st Submission | Loss Plots + Kaggle Submission |
| **P8** | 16-18 Nov | Data Augmentation and improvement | Baseline vs Augmented Comparison |
| **P9** | 23-25 Nov | Inference, Threshold and Errors | 5 normal images + 2 error cases |
| ▶ **P10** | 30 Nov-2 Dec | TTA, Final Submission and Defense | Best Dice Score + Oral Defense |

> **Golden Rule:** According to Art. 7.5 of the UPCT Assessment Regulations, attending and validating the Checkpoint in the classroom is mandatory to pass the practice.


# Practice 10: TTA, Final Submission and Project Defense
## Single session (30 Nov Group A / 2 Dec Group B)

### Session objectives:
1. Implement **Test Time Augmentation (TTA)** to improve the robustness of the predictions on the test set.
2. Generate the definitive **Final Submission** for the Kaggle Leaderboard.
3. Structure and rehearse the **Oral Defense** of the project.

### The Kaggle "Trick": TTA
In real competitions, the model does not predict just once. The original image, its flipped version, and sometimes rotated versions are passed through it. The probabilities are then averaged. This reduces variance and usually raises the Dice Score by between 1% and 3%.

> **CHECKPOINT P10:** Show the professor:
> 1. The TTA code working.
> 2. The screenshot of your final position on the Kaggle Leaderboard.
> 3. The outline of your oral presentation.

## Block 10.1: Test Time Augmentation — the Flip Side of Data Augmentation
### TTA is the counterpart of Data Augmentation, but at inference time

In Practice 8 you trained with `HorizontalFlip(p=0.5)` to force the model to be **invariant** to horizontal flipping: so that it learns that a tumor is still the same tumor whether it is on the left or the right side of the image. TTA exploits that learned invariance, but at the moment of predicting rather than training:

| | Data Augmentation (P8) | Test Time Augmentation (P10) |
|---|---|---|
| When is it applied? | During training | During inference |
| What does it transform? | The training images | The test image, several times |
| What does it achieve? | The model *learns* to be invariant | It exploits that invariance to *average* several opinions of the same model |
| Do the weights change? | Yes, it is trained with them | No, the model is already fixed |

> **Key idea:** if the model never saw horizontally flipped examples during training, there is no reason to expect its prediction on the flipped image to be reliable — TTA only works because Practice 8 prepared the model for this.

### Why the flip of the prediction must be undone before averaging

When you flip the input image and pass it through the model, the mask you get also comes out flipped: a tumor that is on the left in the original image appears "on the right" of the canvas in the prediction on the flipped image. If you averaged that prediction directly with the original one (without restoring its orientation), you would be mixing the pixel `(x, y)` of one with the pixel `(width - x, y)` of the other — two positions that do not represent the same point of the image. That is why the algorithm explicitly asks you to flip the prediction back **before** the average: it is the same principle as Block 7.1 (a geometric transformation must be undone symmetrically so that things stay aligned).

### Averaging probabilities, not already-binarized masks

Notice the exact order of the algorithm: `probs_final = (probs_orig + probs_flip) / 2`, and **only afterwards** `best_threshold` is applied. This is no coincidence — it is the same "threshold once, at the end" principle you saw in Practice 9. If you first binarized each prediction separately and then averaged two masks of 0s and 1s, you would lose the information of *how confident* each view was: a pixel where one view says 0.9 and the other 0.6 (clear tumor in both) and another where one says 0.51 and the other 0.49 (doubtful boundary) can give the same result if you binarize before averaging, but they are very different situations if you average the probabilities first.

### The cost of TTA: inference time, not training time

Data Augmentation and TTA have their computational cost on opposite sides of the pipeline:

| Technique | Where it costs more time |
|---------|----------------------------|
| Data Augmentation | Training (each epoch processes different variations) |
| TTA | Inference (each test image is predicted 2 times: original + flipped) |

With 234 test images and only 2 views per image, the extra cost is affordable in Colab. If instead of 1 flip you used 4 or 8 different transformations, the inference time would grow proportionally — TTA is a trade-off decision between available time and expected Dice improvement.

### Why horizontal flip and not any other transformation

In Block 8.1 you established that a 90° rotation or a vertical flip are not clinically valid transformations for breast ultrasound (they change the anatomical orientation in a way that would never happen in a real acquisition) — and that is why the augmentation pipeline never used them. The same rule applies here: TTA should only use transformations the model **was trained to live with**. Applying TTA with a 90° rotation, for example, would average a reliable prediction (the original one) with a prediction on an input the model never learned to interpret — at best it does not help, at worst it worsens the result.

> **Question to think about:** the statement says that TTA "usually" raises the Dice by between 1% and 3%, not that it always guarantees it. Based on the above, in which specific circumstance would you expect TTA with horizontal flip *not* to improve, or even to worsen, the result?

### Quick summary

| Concept | Main idea |
|----------|-----------------|
| TTA vs Data Augmentation | DA trains invariance; TTA exploits it at inference, without touching the weights |
| Undo the flip | The prediction on the flipped image also comes out flipped; it must be reverted before comparing pixel by pixel |
| Average before thresholding | Averaging probabilities preserves confidence nuances that averaging binary masks loses |
| Cost | Extra time at inference (proportional to the number of views), not at training |
| Valid transformations | Only those the model saw during training (Block 8.1) |

## Task 10.1: Test Time Augmentation (TTA) Implementation
### What is TTA?
During training we use *Data Augmentation* so that the model sees more data.
During inference (test), we use *Test Time Augmentation* so that the model "sees" the same image from different perspectives and averages its confidence.

### The algorithm:
1. Take the original test image.
2. Predict the probability mask (`probs_orig`).
3. Flip the image horizontally (`flip_img`).
4. Predict the mask of the flipped image (`probs_flip`).
5. Flip the prediction back to the original orientation.
6. **Average:** `probs_final = (probs_orig + probs_flip) / 2`.
7. Apply the `best_threshold` on `probs_final`.

### Instructions:
1. Define a function `predict_with_tta(model, img_tensor, device, threshold)`.
2. Inside it, make the normal prediction.
3. Apply `torch.flip(img_tensor, [-1])` to flip the image.
4. Make the second prediction and flip the result back with `torch.flip()`.
5. Average both outputs and binarize with the threshold.

In [ ]:
# ============================================================
# TASK 10.1: INFERENCE FUNCTION WITH TTA
# ============================================================

# WRITE YOUR CODE HERE

def predict_with_tta(model, img_tensor, device, threshold=0.5):
    """
    Performs inference using Test Time Augmentation (Horizontal Flip).
    """
    model.eval()
    with torch.no_grad():
        # 1. Original prediction
        # Your code here...

        # 2. Flip the image and predict
        # img_flipped = torch.flip(img_tensor, [-1])
        # Your code here...

        # 3. Flip the prediction back
        # Your code here...

        # 4. Average and threshold
        # Your code here...

    return mask_pred

print("TTA function defined")

## Block 10.2: The Last Details Before the Definitive Submission
### Why `dtype=torch.float32` is "KEY"

The code explicitly sets `torch.tensor(img_norm, dtype=torch.float32)`. This is not a generic precaution — without it, you would run into a real and quite cryptic error the first time it happens to you.

`numpy` uses `float64` (double precision) by default in many arithmetic operations, while your model weights are `float32` (single precision) — it is the standard in neural networks, because doubling the precision does not improve the result and does double the memory used. If you build the input tensor without fixing the dtype and `numpy` sneaks a `float64` into some intermediate step, PyTorch will return something like `RuntimeError: expected scalar type Float but found Double` at the first convolutional layer — the model cannot multiply `float32` weights by a `float64` input.

> **Key idea:** explicitly fixing `dtype=torch.float32` when creating the tensor is safer than trusting that all previous operations (`.astype(np.float32)`, subtractions, divisions) keep the precision you expect at every step.

### Nesting `torch.no_grad()` is not an error

The comment `# Optional, the function already handles it, but good practice` points at something real: `predict_with_tta` already wraps its contents in `torch.no_grad()` (Task 10.1). Putting a second `with torch.no_grad():` around the call, in this task's loop, is redundant but **is not an error** — PyTorch context managers can be nested without any problem; the second `no_grad()` simply confirms a condition that was already active. It is a reasonable defensive practice: if in the future someone modifies `predict_with_tta` and forgets the internal `no_grad()`, the outer one would still protect the inference loop.

### The resize and the second threshold, once more

In the post-processing section (`mask_resized = cv2.resize(mask_np, ...)` followed by `mask_binary = (mask_resized > 0.5)`) you will recognize exactly the pattern from Block 9.3: `predict_with_tta` returns a mask already binarized at `IMG_SIZE` resolution; when you resize it to the original size with `cv2.resize`, the interpolation reintroduces intermediate values at the borders, and that is why that second `> 0.5` is needed before passing it to `mask_to_rle`. The pattern does not change because you use TTA — only the mask you resize now comes from averaging two views instead of one.

### The complete journey of a submission

At this point, each submission you have uploaded to Kaggle represents one more layer on top of the previous one:

```
submission.csv          (P7):  baseline model, fixed threshold 0.5
submission_final.csv    (P9):  best model + calibrated best_threshold
submission_TTA.csv      (P10): best model + best_threshold + average of 2 views (TTA)
```

Each file should, in principle, beat (or at least match) the previous one on the Leaderboard — if it does not, it is a useful signal for Task 9.3 (reviewing what is failing) rather than something to hide in the oral defense.

### Quick summary

| Concept | Main idea |
|----------|-----------------|
| `dtype=torch.float32` | Avoids the error of mixing `float64` (numpy) with the model's `float32` weights |
| Nested `torch.no_grad()` | Safe and redundant, not an error — reinforces a protection that already existed |
| Resize + second threshold | Same pattern as Block 9.3: the interpolation reintroduces intermediate values at the borders |
| Submission progression | Each CSV adds an improvement over the previous one: baseline → optimal threshold → TTA |

## Task 10.2: Generating the Final Submission with TTA
This is the moment of truth. We are going to use the TTA function to generate our last `submission.csv`.

### Instructions:
1. Iterate over the images in `test/images`.
2. Preprocess the image (remember the `dtype=torch.float32` to avoid type errors).
3. Call `predict_with_tta()` using your `best_threshold`.
4. Resize the mask to the original size and convert it to RLE.
5. Save the DataFrame as **`submission_TTA.csv`** and upload it to Kaggle.

> **Hint:** TTA doubles the inference time, but on a dataset of 234 images in Colab it will only take a few extra minutes. It is worth it!

> **CHECKPOINT P10.1:** Download `submission_TTA.csv` and upload it to Kaggle. Show the professor your new position on the Leaderboard.

In [ ]:
# ============================================================
# TASK 10.2: FINAL SUBMISSION WITH TTA
# ============================================================
import pandas as pd

test_img_dir = DATA_DIR / 'test' / 'images'
test_images = sorted(list(test_img_dir.glob('*.png')))
results = []

print(f"Generating Final Submission with TTA (Threshold = {best_threshold:.2f})...")
print(f"Total images: {len(test_images)}")

with torch.no_grad(): # Optional, the function already handles it, but good practice
    for idx, img_path in enumerate(test_images):
        # 1. Load and preprocess
        img = cv2.imread(str(img_path))
        img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        original_h, original_w = img_rgb.shape[:2]

        img_resized = cv2.resize(img_rgb, (IMG_SIZE, IMG_SIZE))
        img_norm = img_resized.astype(np.float32) / 255.0

        # ImageNet normalization
        mean = np.array([0.485, 0.456, 0.406], dtype=np.float32)
        std = np.array([0.229, 0.224, 0.225], dtype=np.float32)
        img_norm = (img_norm - mean) / std

        # KEY: dtype=torch.float32
        img_tensor = torch.tensor(img_norm, dtype=torch.float32).permute(2, 0, 1).unsqueeze(0).to(DEVICE)

        # 2. Predict with TTA
        # Your code here...

        # 3. Post-process and RLE
        # mask_np = mask_pred.squeeze().cpu().numpy()
        # mask_resized = cv2.resize(mask_np, (original_w, original_h))
        # mask_binary = (mask_resized > 0.5).astype(np.uint8)
        # rle = mask_to_rle(mask_binary)
        # results.append({'Id': img_path.name, 'Expected': rle})

        if (idx + 1) % 50 == 0:
            print(f"   Progress: {idx + 1}/{len(test_images)}")

# Save CSV
submission_df = pd.DataFrame(results)
submission_df.to_csv('submission_TTA.csv', index=False)
print("Final Submission generated! (submission_TTA.csv)")

## Block 10.3: Prepare the Questions, not Just the Slides
### Why the oral defense weighs as much as the code

A model that works but that its author cannot explain is, in professional practice, a model that cannot be defended before an ethics committee, a skeptical radiologist, or a reviewer of a scientific article. The oral defense is not an added formality at the end of the project — it is the verification that you understood the decisions you made, not just that you copied code that works.

### The question that breaks an unprepared defense: "why?"

The assessment criteria ask for "critical thinking: not just reading numbers, but explaining why the model fails or succeeds". In practice, this means the professor is not going to ask "what Dice Score did you get?" (that is already on the screen) but "why that threshold and not 0.5?", "why does this image fail and this other one not?". If you only memorize the results without understanding the decisions behind them, those questions will catch you blank in front of the professor — which is exactly the moment that counts for the grade.

### A battery of questions to test yourselves before the defense

Review these questions on your own before the defense. Each one corresponds to a Block you have already worked on — if you cannot answer one confidently, that is the block you should reread:

| Question | Block where the answer is |
|----------|-----------------------------------|
| Why do you normalize with the ImageNet mean/std and not just divide by 255? | Block 7.1 (Dataset and DataLoader) |
| Why does the loss function combine Dice + BCE instead of using only one? | Block 7.2 (U-Net and Loss Function) |
| What is the difference between `model.train()` and `model.eval()`, and why does it matter? | Block 7.3 (Training Loop) |
| Why can you not compute your real Dice on the test set? | Block 7.4 (From Training to Submission) |
| Why is a 90° rotation not a valid augmentation here? | Block 8.1 (Data Augmentation) |
| Why do you need to "denormalize" the image in order to visualize it? | Block 8.2 (Verify Visually) |
| Why did you reinitialize the model instead of continuing to train the baseline? | Block 8.3 (Repeat an Experiment) |
| Why is looking only at the final Val Dice not enough to conclude that a model is better? | Block 8.4 (Reading a Comparison) |
| Why is the threshold calibrated on validation and not on train or test? | Block 9.1 (The Threshold as a Hyperparameter) |
| Why can a "normal" image never produce a False Negative? | Block 9.2 (Reading the Errors) |
| Why must the mask be resized carefully before the RLE? | Block 9.3 (Closing the Loop) |
| Why should TTA with horizontal flip work, but one with a 90° rotation not? | Block 8.1 (Data Augmentation) |

> **Key idea:** if a classmate from another group asked you these questions without warning, could you answer without looking at the notebook? That is the real test of whether you are ready for the defense.

### How to answer when you do not know something

If the professor asks something you do not remember precisely, the worst possible answer is to invent something that sounds good but is wrong — an experienced evaluator detects it right away, and penalizes the attempt to "fill in" more than honesty. It is better to say "I am not sure of the exact reason, but what we observed was..." and describe what you did see in your own results. Technical honesty is also part of what is assessed as "technical clarity".

### Quick summary

| Concept | Main idea |
|----------|-----------------|
| Why the defense matters | A model you cannot explain is not professionally defensible |
| The key question | It is not "what result did you get" but "why did you make that decision" |
| Prior self-assessment | Review the per-block question table before the defense, not just the slides |
| When facing a real doubt | Honesty about what was observed, better than an invented justification |

## Task 10.3: Preparation of the Oral Defense
You have done the work of biomedical engineers. Now it is time to communicate it. The oral defense is 50% of the grade for Practice 10.

### Recommended presentation structure (3-5 minutes per person):

1. **Clinical Context (1 min):**
   * What is the BUSI dataset? Why is segmenting breast tumors important?
   * What clinical challenge do the "normal" images pose?
2. **EDA and Preprocessing (1 min):**
   * Class distribution.
   * Explanation of the RLE format and why it is necessary.
3. **Architecture and Training (2 min):**
   * Why U-Net? (Mention Skip Connections).
   * Loss function: Why do we combine Dice + BCE?
   * Data Augmentation strategy: What worked and what did not? (Show comparative plots).
4. **Error Analysis and Threshold (1.5 min):**
   * Show the threshold optimization plot.
   * Show 1 success case and 1 failure case (False Positive/Negative) and explain *clinically* why it happened.
5. **Conclusions and TTA (0.5 min):**
   * How much did the final score improve with TTA?
   * What would you do if you had one more month for the project?

### Assessment Criteria:
* **Technical clarity:** Correct use of terminology (Dice, BCE, Overfitting, TTA).
* **Critical thinking:** Not just reading numbers, but explaining *why* the model fails or succeeds.
* **Teamwork:** Both members of the group must speak.

> **CHECKPOINT P10.2:** Show the professor the outline of your presentation before leaving class. Good luck in the defense!